In [15]:
import pandas as pd
import numpy as np
import pickle
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

In [16]:
# Load the processed dataset
df = pd.read_csv('processed_online_course_data.csv')
print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

Dataset loaded with 100000 rows and 15 columns.


,user_id,course_id,course_name,instructor,course_duration_hours,certification_offered,difficulty_level,rating,enrollment_numbers,course_price,feedback_score,study_material_available,time_spent_hours,previous_courses_taken,completion_ratio
0,15796,9366,Python for Beginners,Emma Harris,39.1,1,0,5.0,21600,317.50,0.797,1,17.60,4,0.450128
1,861,1928,Cybersecurity for Professionals,Alexander Young,36.3,1,0,4.3,15379,40.99,0.770,1,28.97,9,0.798072
2,38159,9541,DevOps and Continuous Deployment,Dr. Mia Walker,13.4,1,0,3.9,6431,380.81,0.772,1,52.44,4,3.913433
3,44733,3708,Project Management Fundamentals,Benjamin Lewis,58.3,1,0,3.1,48245,342.80,0.969,0,22.29,6,0.382333
4,11285,3361,Ethical Hacking Masterclass,Daniel White,30.8,1,0,2.8,34556,381.01,0.555,1,22.01,5,0.714610


In [17]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Training set size: {len(train_df)}")
print(f"Testing set size: {len(test_df)}")

Training set size: 80000
Testing set size: 20000


In [18]:
# 1. Create the User-Item Pivot Table
user_item_matrix = train_df.pivot_table(index='user_id', columns='course_name', values='rating').fillna(0)

# 2. Calculate User Mean Ratings for Normalization
user_mean_ratings = user_item_matrix.mean(axis=1)

# 3. Normalize the matrix (Center the ratings around zero)
user_item_matrix_norm = user_item_matrix.sub(user_mean_ratings, axis=0)

# 4. Initialize and Train SVD
# We use 50 components as a standard latent factor count
n_components = min(50, min(user_item_matrix_norm.shape) - 1)
svd_model = TruncatedSVD(n_components=n_components, random_state=42)
latent_matrix = svd_model.fit_transform(user_item_matrix_norm)

# 5. Reconstruct predicted (normalized) ratings
predicted_ratings_norm = svd_model.inverse_transform(latent_matrix)
predicted_ratings_df = pd.DataFrame(predicted_ratings_norm, 
                                    index=user_item_matrix.index, 
                                    columns=user_item_matrix.columns)

print("SVD model trained on normalized data successfully.")

SVD model trained on normalized data successfully.


In [19]:
def predict_rating(user_id, course_name):
    # If user and course are in training data, provide SVD prediction
    if user_id in predicted_ratings_df.index and course_name in predicted_ratings_df.columns:
        pred_norm = predicted_ratings_df.loc[user_id, course_name]
        user_mean = user_mean_ratings.loc[user_id]
        return pred_norm + user_mean
    else:
        # Fallback to global average if user or course is unknown
        return train_df['rating'].mean()

# Apply prediction function to test set
test_df['predicted_rating'] = test_df.apply(lambda x: predict_rating(x['user_id'], x['course_name']), axis=1)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(test_df['rating'], test_df['predicted_rating']))
print(f"Matrix Factorization (SVD) RMSE: {rmse:.4f}")

Matrix Factorization (SVD) RMSE: 3.4567


In [20]:
# Prepare full matrix
full_matrix = df.pivot_table(index='user_id', columns='course_name', values='rating').fillna(0)
full_means = full_matrix.mean(axis=1)
full_norm = full_matrix.sub(full_means, axis=0)

# Train final model
final_svd = TruncatedSVD(n_components=n_components, random_state=42)
final_svd.fit(full_norm)

# Save the exact files needed by app.py
with open('svd_norm_final.pkl', 'wb') as f:
    pickle.dump(final_svd, f)

with open('user_mean_ratings_final.pkl', 'wb') as f:
    pickle.dump(full_means, f)

print("Deployment files 'svd_norm_final.pkl' and 'user_mean_ratings_final.pkl' are ready.")

Deployment files 'svd_norm_final.pkl' and 'user_mean_ratings_final.pkl' are ready.
